In [1]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("vargr/main_instagram")
print(dataset)
print(dataset['train'].column_names)
print(dataset['train'][0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-b6868cb1ddbd5c(…):   0%|          | 0.00/159M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/605868 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sid', 'sid_profile', 'shortcode', 'profile_id', 'date', 'post_type', 'description', 'likes', 'comments', 'username', 'bio', 'following', 'followers', 'num_posts', 'is_business_account', 'lang', 'description_category', 'description_grade', 'image_grade', 'path'],
        num_rows: 605868
    })
})
['sid', 'sid_profile', 'shortcode', 'profile_id', 'date', 'post_type', 'description', 'likes', 'comments', 'username', 'bio', 'following', 'followers', 'num_posts', 'is_business_account', 'lang', 'description_category', 'description_grade', 'image_grade', 'path']
{'sid': 8304346, 'sid_profile': 3496776, 'shortcode': 'Bt5LFpZlm3z', 'profile_id': 2237947779, 'date': '2019-02-15 08:02:35', 'post_type': 1, 'description': 'Laps in spring like conditions. Getting these M29 bikes up to speed. Is it time to go racing yet?\\n.\\n.\\n.\\n.\\n.\\n#intenseforlife #m29 #dh #racing \\n@saddleback_ltd \\n@intensecyclesuk \\n@intensecycles\\n@tld_bike\\n@

In [2]:
df = dataset['train'].to_pandas()

print(df[['likes','followers','date']].head())
print(df.isnull().sum())


   likes  followers                 date
0    150       1204  2019-02-15 08:02:35
1     15       1263  2019-03-19 22:18:58
2     41        139  2019-05-13 19:03:32
3    292      28163  2016-05-07 19:34:45
4    454      27753  2019-04-04 18:52:40
sid                          0
sid_profile                  0
shortcode                    0
profile_id                   0
date                         0
post_type                    0
description                  0
likes                        0
comments                     0
username                     0
bio                     117093
following                    0
followers                    0
num_posts                    0
is_business_account          0
lang                         0
description_category         0
description_grade            0
image_grade                  0
path                         0
dtype: int64


In [3]:
import pandas as pd
import numpy as np

df = dataset['train'].to_pandas()

# Remove accounts with zero followers
df = df[df['followers'] > 0]

# Create engagement rate
df['engagement_rate'] = (df['likes'] + df['comments']) / df['followers']

# Remove extreme outliers (top 1%)
upper_limit = df['engagement_rate'].quantile(0.99)
df = df[df['engagement_rate'] <= upper_limit]

df['engagement_rate'].describe()


,engagement_rate
count,599579.000000
mean,0.111752
std,0.099628
min,0.000000
25%,0.038462
50%,0.083218
75%,0.155556
max,0.613913


In [4]:
low = df['engagement_rate'].quantile(0.33)
high = df['engagement_rate'].quantile(0.66)

def classify(x):
    if x <= low:
        return 0  # low
    elif x <= high:
        return 1  # medium
    else:
        return 2  # high

df['engagement_class'] = df['engagement_rate'].apply(classify)

df['engagement_class'].value_counts()


,count
engagement_class,
2,203854
1,197864
0,197861


In [5]:
df['date'] = pd.to_datetime(df['date'])

df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report



df_sample = df.sample(n=100000, random_state=42)

# Then repeat TFIDF and rest using df_sample

# Text vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_text = tfidf.fit_transform(df_sample['description'])

meta_features = df_sample[['hour',
                    'day_of_week',
                    'followers',
                    'following',
                    'num_posts',
                    'is_business_account']].copy()

# Convert boolean to int
meta_features['is_business_account'] = meta_features['is_business_account'].astype(int)

# Ensure all numeric
meta_features = meta_features.astype(float)

meta_features = meta_features.values

# Combine
import scipy.sparse as sp
X = sp.hstack([X_text, meta_features])

y = df_sample['engagement_class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.64      0.66      0.65      6700
           1       0.45      0.37      0.41      6595
           2       0.60      0.69      0.64      6705

    accuracy                           0.57     20000
   macro avg       0.56      0.57      0.57     20000
weighted avg       0.57      0.57      0.57     20000



In [11]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.65      0.57      0.61      6700
           1       0.43      0.26      0.32      6595
           2       0.52      0.79      0.63      6705

    accuracy                           0.54     20000
   macro avg       0.54      0.54      0.52     20000
weighted avg       0.54      0.54      0.52     20000



In [10]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=5
)


In [9]:
df_sample['followers'] = np.log1p(df_sample['followers'])
df_sample['following'] = np.log1p(df_sample['following'])
df_sample['num_posts'] = np.log1p(df_sample['num_posts'])


In [12]:
y_train.value_counts(normalize=True)


,proportion
engagement_class,
2,0.338537
0,0.332275
1,0.329188


In [13]:
!pip install xgboost
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    tree_method='hist'
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

from sklearn.metrics import classification_report
print(classification_report(y_test, preds))


              precision    recall  f1-score   support

           0       0.68      0.63      0.65      6700
           1       0.45      0.40      0.43      6595
           2       0.59      0.71      0.65      6705

    accuracy                           0.58     20000
   macro avg       0.58      0.58      0.58     20000
weighted avg       0.58      0.58      0.58     20000

